In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import os
from transformers import TextStreamer
from tqdm.auto import tqdm

In [11]:
model_name = "microsoft/phi-2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

c:\Users\Andrius\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Andrius\.cache\huggingface\hub\models--microsoft--phi-2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

In [12]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype = torch.bfloat16, #to make memory efficient
    trust_remote_code = False #to ensure security when loading code from the model repository 
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

PhiForCausalLM(
  (model): PhiModel(
    (embed_tokens): Embedding(51200, 2560)
    (layers): ModuleList(
      (0-31): 32 x PhiDecoderLayer(
        (self_attn): PhiAttention(
          (q_proj): Linear(in_features=2560, out_features=2560, bias=True)
          (k_proj): Linear(in_features=2560, out_features=2560, bias=True)
          (v_proj): Linear(in_features=2560, out_features=2560, bias=True)
          (dense): Linear(in_features=2560, out_features=2560, bias=True)
        )
        (mlp): PhiMLP(
          (activation_fn): NewGELUActivation()
          (fc1): Linear(in_features=2560, out_features=10240, bias=True)
          (fc2): Linear(in_features=10240, out_features=2560, bias=True)
        )
        (input_layernorm): LayerNorm((2560,), eps=1e-05, elementwise_affine=True)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
    )
    (rotary_emb): PhiRotaryEmbedding()
    (embed_dropout): Dropout(p=0.0, inplace=False)
    (final_layernorm): LayerNorm((2560,), eps=1

In [ ]:
# trying annotations
prompt = (
    "Can you annotate this text with BIOs tags for named entities: ORG, PER, LOC, MISC? "
    "Additionally, annotate borrower and lender entities with PER tags.\n"
    "Here is the text example:\n"
    "Subordinated\tO\n"
    "Loan\tO\n"
    "Agreement\tO\n"
    "-\tO\n"
    "Silicium\tI-ORG\n"
    "de\tI-ORG\n"
    "Provence\tI-ORG\n"
    "SAS\tI-ORG\n"
    "and\tO\n"
    "Evergreen\tI-ORG\n"
    "Solar\tI-ORG\n"
    "Inc\tI-ORG\n"
    "Generate annotations for this text in the same format:\n"
    "Dated March 31, 2007 Thinkplus Investments Limited (as the Lender) AND Airland International Limited Bizexpress Limited (as the Borrower) Loan Agreement Contents."
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

generated_ids = model.generate(
    inputs["input_ids"],
    max_new_tokens=300,
    do_sample=True,
)

decoded = tokenizer.decode(generated_ids[0], skip_special_tokens=True)


print(decoded)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


finished


In [66]:
!pip install faker

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 2.0/2.0 MB 16.0 MB/s eta 0:00:00


In [67]:
from faker import Faker

fake = Faker()

lender = fake.company()
lender_address = fake.address().replace('\n', ', ')
borrower = fake.company()
borrower_address = fake.address().replace('\n', ', ')
agreement_date = fake.date_between(start_date='-10y', end_date='today').strftime("%B %d, %Y")

print(f"Agreement Date: {agreement_date}")
print(f"Lender: {lender}, Address: {lender_address}")
print(f"Borrower: {borrower}, Address: {borrower_address}")

Agreement Date: July 07, 2025
Lender: Kim, Perry and Smith, Address: 19368 Craig Alley, Staceyside, ND 82505
Borrower: Fisher and Sons, Address: 0097 Archer Course Suite 202, Daletown, GU 70217


In [70]:
from faker import Faker
from transformers import AutoTokenizer, AutoModelForCausalLM

fake = Faker()

lender = fake.company()
lender_address = fake.address().replace('\n', ', ')
borrower = fake.company()
borrower_address = fake.address().replace('\n', ', ')
agreement_date = fake.date_between(start_date='-10y', end_date='today').strftime("%B %d, %Y")

topic = (
    f"Loan Agreement\n"
    f"Lender: {lender}, {lender_address}\n"
    f"Borrower: {borrower}, {borrower_address}\n"
    f"Agreement Date: {agreement_date}\n"
)

sections = [
    "0. INTRODUCTION",
    "1. PARTIES",
    # "2. DEFINITIONS",
    # "3. LENDING DISCLOSURE",
    # "4. LOAN TERMS",
    # "5. REPAYMENT TERMS",
    # "6. INTEREST RATES AND FEES",
    # "7. COLLATERAL",
    # "8. COVENANTS",
    # "9. DEFAULT AND REMEDIES",
    # "10. MISCELLANEOUS PROVISIONS"
]

output_file = "synthetic/generated_contract.txt"
contract_so_far = ""  # This accumulates the contract's text as context.

with open(output_file, "w", encoding="utf-8") as f:
    f.write("=== SYNTHETIC CONTRACT GENERATION ===\n\n")
    f.write(f"{topic}\n")
    f.write("="*40 + "\n\n")

for section in sections:
    # Build the prompt using prior generated contract content for context (trim if very long!)
    # For small models, keep context window in mind (e.g. use "contract_so_far[-1500:]" for long contracts)
    prompt = (
        f"{topic}\n{contract_so_far[-1500:]}\n"  # Only use last ~1500 chars for context to avoid overflow.
        f"{section}\n"
        "Continue the contract with this section, following legal language and style.\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output_ids = model.generate(
        inputs["input_ids"], max_new_tokens=400, do_sample=True, pad_token_id=tokenizer.eos_token_id
    )
    decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    
    # Remove prompt echo (if present): get only the new text after the prompt
    if decoded.startswith(prompt):
        section_text = decoded[len(prompt):].strip()
    else:
        section_text = decoded.strip()

    # For the next iteration, accumulate the contract so far
    contract_so_far += f"\n\n{section}\n{section_text}"

    # Write to file with annotation
    with open(output_file, "a", encoding="utf-8") as f:
        f.write(f"\n\n--- Prompt for {section} ---\n")
        f.write(prompt.strip() + "\n")
        f.write(f"--- Output for {section} ---\n")
        f.write(section_text)
        f.write("\n" + "="*40 + "\n")

print(f"Done! See the generated contract in: {output_file}")

KeyboardInterrupt: 